# MaxCut

MaxCut splits the nodes of a graph $G$ into two groups so that as many edges as possible join the groups. Spin $\sigma_i = \pm 1$ says which group node $i$ joins, and the number of cut edges is

$$
    C = \frac 12 \sum_{(i, j) \in E(G)} (1 - \sigma_i \sigma_j),
$$

where $E(G)$ is the set of edges of $G$. The QAOA maximizes $C$.

In [ ]:
using QAOA
import MFAOA

using PyPlot
# PyPlot.plt.style.use("paper.mplstyle")
using PyCall
nx = pyimport("networkx");

## QAOA

__Defining the problem by hand:__

In [ ]:
N = 4
graph = nx.cycle_graph(N)

figure(figsize=(3, 2))
nx.draw(graph, with_labels=true)

Note that we have to __shift the edges by 1__ when going from Python to Julia. In Ising form, every edge $(i, j)$ has the coupling $J_{ij} = -1/2$, and the objective has the offset $|E(G)|/2$:

In [ ]:
edges = [edge .+ (1, 1) for edge in graph.edges]
J = zeros(N, N)
for (i, j) in edges
    J[i, j] = J[j, i] = -1/2
end
max_cut_problem = IsingProblem(zeros(N), J; offset = length(edges) / 2, sense = :max)

__Using the wrapper function:__

In [ ]:
max_cut_problem = max_cut(N, edges)

__Optimizing the angles:__ `optimize_schedule` tunes the angles of `p` layers with the BFGS method and exact derivatives. It returns the angles and the average cut they give:

In [ ]:
p = 1
angles, average_cut = optimize_schedule(max_cut_problem, p)

The probability of every assignment after the QAOA with these angles. Each label lists the bits of qubits 1 to $N$:

In [ ]:
probs = probabilities(max_cut_problem, angles)

xlabels = []
for bstr in digits.(0:2^N-1, base=2, pad=N)
    push!(xlabels, "\$|" * prod([string(b) for b in bstr]) * "\\rangle\$")
end

figure(figsize=(5, 3.2))
ax = subplot(111)
bar(0:2^N-1, probs)
ax.set_xticks(0:2^N-1)
ax.set_xticklabels(xlabels, rotation=90)
minorticks_off()
tight_layout()

The states `5` $\equiv |1010\rangle$ and `10` $\equiv |0101\rangle$ are indeed the correct solutions!

## Mean-Field AOA with MFAOA.jl

The [mean-field approximate optimization algorithm](https://doi.org/10.1103/PRXQuantum.4.030335) (mean-field AOA) replaces every qubit by a classical spin vector $\mathbf{n}_i = (n_i^x, n_i^y, n_i^z)$ and runs a QAOA-like schedule on these spins. The package [MFAOA.jl](https://github.com/Quicopt/MFAOA.jl) implements it. It describes problems and angles in the same way as QAOA.jl and exports many of the same names, so we load it with `import`.

The MaxCut objective has no local fields, so flipping every spin leaves it unchanged. The mean-field AOA cannot break this symmetry by itself, so we fix the last spin to $\sigma_N = +1$. Its couplings to the other spins then act on them as local fields $h_i = J_{iN}$:

In [ ]:
h_fixed = J[1:N-1, N]        # local fields from the fixed spin σ_N = +1
J_fixed = J[1:N-1, 1:N-1]    # couplings among the remaining spins
mf_problem = MFAOA.IsingProblem(h_fixed, J_fixed; sense = :max)

__Following the spins through the schedule:__ we take MFAOA's linear schedule with $p = 1000$ steps and record every spin vector after each step.

In [ ]:
p = 1000
mf_schedule = MFAOA.linear_schedule(p, 0.5)

workspace = MFAOA.Workspace(mf_problem)
spins = workspace.state                 # spins.x[i], spins.y[i], spins.z[i] are the components of n_i
history = [[spins.x spins.y spins.z]]   # history[k][i, :] is n_i after k - 1 steps
for k in 1:p
    MFAOA.step!(workspace, mf_problem, mf_schedule.β[k], mf_schedule.γ[k])
    push!(history, [spins.x spins.y spins.z])
end

In [ ]:
# plot x, y, and z of all spins that are not fixed
times = range(0, 1, p + 1)

figure(figsize=((N - 1) * 2.2, 2))
for i in 1:N-1
    subplot(1, N - 1, i)
    for (component, label) in enumerate(["x", "y", "z"])
        plot(times, [n[i, component] for n in history], label=label)
    end
    xlim(0, 1)
    ylim(-1, 1)
    xlabel("t/T")
    ylabel("n_" * string(i))
    i == 1 && legend(fontsize=7)
end
tight_layout()

__Reading off the solution:__ the sign of $n_i^z$ at the end of the schedule is the spin $\sigma_i$. `MFAOA.solve` runs the same schedule and returns these spins:

In [ ]:
result = MFAOA.solve(mf_problem, mf_schedule)
σ = vcat(result.σ, 1)        # append the fixed spin σ_N = +1
bits = (1 .+ σ) .÷ 2         # a measured 1 means σ = +1
state = sum(bits[i] * 2^(i - 1) for i in 1:N)

println("solution |", join(bits), "⟩ = state ", state)
println("cut edges: ", objective(max_cut_problem, σ), " of ", length(edges))

This is the state `10` $\equiv |0101\rangle$ that the QAOA found above, and it cuts all four edges. The other solution, `5` $\equiv |1010\rangle$, has every spin flipped, so fixing $\sigma_4 = +1$ excludes it.

## Annealing

With many layers and slowly changing angles, the QAOA becomes a quantum annealing from the mixer to the objective. `linear_schedule(p, τ)` cuts an annealing of duration $p\tau$ into $p$ layers; the duration is measured in units of the problem's `fieldscale`. The two largest cuts are the states `5` and `10`:

In [ ]:
p = 256
τ = 8 / p
probs = probabilities(max_cut_problem, linear_schedule(p, τ))
probs[5 + 1] + probs[10 + 1]

### Second-order schedule

Any two vectors of angles make a `Schedule`. This one shifts the angles $\gamma$ by half a step and shortens the last $\beta$:

In [ ]:
γ = collect(τ .* ((1:p) .- 1/2) ./ p)
β = collect(τ .* (1 .- (1:p) ./ p))
β[p] = τ / (4 * p)
probs = probabilities(max_cut_problem, Schedule(β, γ))
probs[5 + 1] + probs[10 + 1]